<a href="https://colab.research.google.com/github/hiiD4vd/Machine-Learning/blob/main/submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# SCRAPING
!pip install google-play-scraper transformers accelerate torch scikit-learn matplotlib imbalanced-learn

import pandas as pd
import numpy as np
from google_play_scraper import reviews, Sort


APP_ID = 'com.icon.pln123'  # PLN Mobile
COUNT = 25000               # Saya ambil lebih agar setelah dibersihkan tetep diatas 10k

print(f"🔄 Sedang mengambil {COUNT} data dari PLN Mobile... Mohon tunggu.")

result, _ = reviews(
    APP_ID,
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=COUNT,
    filter_score_with=None
)

df_raw = pd.DataFrame(result)
df_raw = df_raw[['content', 'score', 'at']]
df_raw.to_csv('pln_reviews_raw.csv', index=False)

print(f"✅ BERHASIL! Mendapatkan {len(df_raw)} data mentah.")
print("Lanjut ke Cell berikutnya untuk cleaning & training.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.1 MB/s eta 0:00:00
🔄 Sedang mengambil 25000 data dari PLN Mobile... Mohon tunggu.
✅ BERHASIL! Mendapatkan 25000 data mentah.
Lanjut ke Cell berikutnya untuk cleaning & training.


In [ ]:
import pandas as pd
import numpy as np
import torch
import re
import string
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

# Cek GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Perangkat: {device} (Pastikan pakai GPU di Runtime > Change runtime type)")


print("\n[1/5] Memulai Cleaning Data...")
df = pd.read_csv('pln_reviews_raw.csv')

def labeling(score):
    if score < 3: return 0      # Negative
    elif score == 3: return 1   # Neutral
    else: return 2              # Positive

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'@[A-Za-z0-9_]+', '', text) # Hapus username
    text = re.sub(r'#\w+', '', text)           # Hapus hashtag
    text = re.sub(r'\d+', '', text)            # Hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation)) # Hapus tanda baca
    text = text.strip()
    text = re.sub(r'(.)\1{2,}', r'\1\1', text) # Hapus huruf berulang (baguuus -> baguus)
    return text

df['label'] = df['score'].apply(labeling)
df['text_clean'] = df['content'].apply(clean_text)

df_clean = df[df['text_clean'] != ''].copy()
df_clean = df_clean.drop_duplicates(subset=['text_clean'])
df_clean['word_count'] = df_clean['text_clean'].apply(lambda x: len(x.split()))
df_clean = df_clean[df_clean['word_count'] >= 3] # Hapus yang kependekan

print(f"✅ Total Data Bersih: {len(df_clean)}")
if len(df_clean) < 10000:
    print("⚠️ PERINGATAN: Data kurang dari 10.000! Risiko Bintang 5 hilang.")
else:
    print("🌟 Syarat Data > 10.000 Terpenuhi.")

print("\n[2/5] Menyiapkan Data Training...")

# Split 80:20 Standar
X = df_clean['text_clean'].values.reshape(-1, 1)
y = df_clean['label'].values
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Oversampling (HANYA TRAIN)
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(X_train_raw, y_train)

# Format list untuk Tokenizer
X_train_final = [x[0] for x in X_train_resampled]
X_test_final = [x[0] for x in X_test_raw]
y_train_final = y_train_resampled

print("\n[3/5] Training IndoBERT (Bisa memakan waktu 10-20 menit)...")

tokenizer = BertTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
train_encodings = tokenizer(X_train_final, truncation=True, padding=True, max_length=64)
test_encodings = tokenizer(X_test_final, truncation=True, padding=True, max_length=64)

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, y_train_final)
test_dataset = ReviewDataset(test_encodings, y_test)

model = BertForSequenceClassification.from_pretrained('indobenchmark/indobert-base-p1', num_labels=3)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

training_args = TrainingArguments(
    output_dir='./results_pln',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
eval_result = trainer.evaluate()
final_acc_bert = eval_result['eval_accuracy'] * 100

print("\n[4/5] Menjalankan Skema Pembanding (Syarat Variasi)...")

# SKEMA 2: SVM (Split 70:30)
print("   -> Training SVM (Split 70:30)...")
X_svm = df_clean['text_clean'].values
y_svm = df_clean['label'].values
# Variasi Split 70:30
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(X_svm, y_svm, test_size=0.30, random_state=1, stratify=y_svm)

vectorizer_svm = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf_svm = vectorizer_svm.fit_transform(X_train_svm)
X_test_tfidf_svm = vectorizer_svm.transform(X_test_svm)

svm = SVC(kernel='linear', C=1.0)
svm.fit(X_train_tfidf_svm, y_train_svm)
acc_svm = accuracy_score(y_test_svm, svm.predict(X_test_tfidf_svm)) * 100

# SKEMA 3: NAIVE BAYES (Split 80:20)
print("   -> Training Naive Bayes (Split 80:20)...")
# Gunakan data split 80:20 dari awal (X_train_final yang sudah oversampled)
vectorizer_nb = TfidfVectorizer(max_features=5000)
X_train_tfidf_nb = vectorizer_nb.fit_transform(X_train_final)
X_test_tfidf_nb = vectorizer_nb.transform(X_test_final)

nb = MultinomialNB()
nb.fit(X_train_tfidf_nb, y_train_final)
acc_nb = accuracy_score(y_test, nb.predict(X_test_tfidf_nb)) * 100

print("\n" + "="*40)
print("🏆 LAPORAN AKHIR PROYEK ANALISIS SENTIMEN")
print("="*40)
print(f"1. IndoBERT (DL, 80:20)   : {final_acc_bert:.2f}%")
print(f"2. SVM (ML, 70:30)        : {acc_svm:.2f}%")
print(f"3. Naive Bayes (ML, 80:20): {acc_nb:.2f}%")

print("\n--- STATUS SUBMISSION ---")
if final_acc_bert >= 92 and len(df_clean) >= 10000:
    print("✅ MEMENUHI SYARAT BINTANG 5 (Akurasi >92% & Data >10k)")
elif final_acc_bert >= 85:
    print("✅ LULUS (Akurasi >85%)")
else:
    print("⚠️ PERINGATAN: Cek akurasi kembali.")

# Tes Manual
print("\n--- DEMO PREDIKSI (INFERENCE) ---")
def predict_manual(text):
    inputs = tokenizer(clean_text(text), return_tensors="pt", truncation=True, max_length=64).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits).item()
    label_map = {0: 'Negatif 😡', 1: 'Netral 😐', 2: 'Positif 😊'}
    print(f"Teks: '{text}'\nHasil: {label_map[pred]}\n")

predict_manual("Aplikasi sering error, tolong diperbaiki")
predict_manual("Lumayan membantu untuk bayar listrik")
predict_manual("Sangat bagus, fitur lengkap dan cepat")

🚀 Perangkat: cuda (Pastikan pakai GPU di Runtime > Change runtime type)

[1/5] Memulai Cleaning Data...
✅ Total Data Bersih: 18272
🌟 Syarat Data > 10.000 Terpenuhi.

[2/5] Menyiapkan Data Training...

[3/5] Training IndoBERT (Bisa memakan waktu 10-20 menit)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.009500,0.132185,0.981943
2,0.006500,0.133770,0.983858
3,0.000000,0.135945,0.985499



[4/5] Menjalankan Skema Pembanding (Syarat Variasi)...
   -> Training SVM (Split 70:30)...
   -> Training Naive Bayes (Split 80:20)...

🏆 LAPORAN AKHIR PROYEK ANALISIS SENTIMEN
1. IndoBERT (DL, 80:20)   : 98.55%
2. SVM (ML, 70:30)        : 98.45%
3. Naive Bayes (ML, 80:20): 96.22%

--- STATUS SUBMISSION ---
✅ MEMENUHI SYARAT BINTANG 5 (Akurasi >92% & Data >10k)

--- DEMO PREDIKSI (INFERENCE) ---
Teks: 'Aplikasi sering error, tolong diperbaiki'
Hasil: Negatif 😡

Teks: 'Lumayan membantu untuk bayar listrik'
Hasil: Positif 😊

Teks: 'Sangat bagus, fitur lengkap dan cepat'
Hasil: Positif 😊

